In [5]:
# ==============================================================================
# 1. 환경 설정 및 데이터 준비 (Setup & Data Preparation)
# ==============================================================================

!git clone https://github.com/boragu/SRL-Code-Review-Study.git
%cd "/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting"
!mkdir -p Gayoon

import os
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import gdown

USER_NAME = "Gayoon"
PROJECT_DIR = Path(f"/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/{USER_NAME}")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# 작업용 폴더 자동 생성 (exist_ok=True로 중복 생성 방지)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 데이터셋 압축파일 다운로드 및 지정 경로 해제
ZIP_PATH = DATA_DIR / "data.zip"
gdown.download(id="1qUXEalsDiePFDUNuI4ThEihB0Jpr8VEt", output=str(ZIP_PATH), quiet=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")
oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_DIR / "holidays_events.csv", parse_dates=["date"])


# ==============================================================================
# 2. 시계열 피처 생성 및 모델링 (Time Series Feature Engineering & Modeling)
# ==============================================================================

def create_my_pipeline(df, is_train=True):
    # store_nbr 키 기준으로 매장 상세 정보(city, state, type, cluster) 조인
    df = df.merge(stores, on="store_nbr", how="left")

    # date 키 기준으로 유가 데이터 조인 후 결측치는 전진 채우기(ffill) 및 후진 채우기(bfill)로 보정
    df = df.merge(oil.rename(columns={"dcoilwtico": "oil_price"}), on="date", how="left")
    df["oil_price"] = df["oil_price"].ffill().bfill()

    # datetime 속성 이용한 주기성 시간 피처 분할 추출
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # 에콰도르 소매업 특성인 월급날(15일, 30/31일) 구매력 급증 현상 이진 피처화
    df["is_payday"] = df["day"].isin([15, 30, 31]).astype(int)

    # transferred==True 인 이벤트(이월된 휴일)를 정제하고 날짜 중복 제거
    holidays_clean = holidays[~holidays["transferred"]][["date", "type"]].drop_duplicates("date")
    holidays_clean["is_holiday"] = 1

    # 날짜 기준 휴일 여부 매핑 및 미지정일(NaN)을 0으로 변환
    df = df.merge(holidays_clean[["date", "is_holiday"]], on="date", how="left")
    df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

    return df

# 전처리 파이프라인 적용
train_df = create_my_pipeline(train, is_train=True)
test_df = create_my_pipeline(test, is_train=False)

# 시계열 인덱스 순서 유지를 위한 매장, 상품군, 날짜 오름차순 정렬
train_df = train_df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# Sales 값의 우편향(Right-skewed) 분포를 완화하기 위한 로그 변환 (log(1 + x))
train_df["log_sales"] = np.log1p(train_df["sales"])

# 테스트 데이터 기간(16일)의 Data Leakage 방지를 위해 최소 16일 이상의 시차(Lag) 설정
train_df["lag_16"] = train_df.groupby(["store_nbr", "family"])["log_sales"].shift(16)
train_df["lag_30"] = train_df.groupby(["store_nbr", "family"])["log_sales"].shift(30)

# 최근 일자 매출 데이터에 더 큰 가중치를 부여하는 지수 이동 평균(EWMA) 생성
train_df["ewm_sales_7"] = train_df.groupby(["store_nbr", "family"])["log_sales"].transform(
    lambda x: x.shift(16).ewm(span=7).mean()
)

# 학습 데이터셋 컬럼 및 범주형(Categorical) 피처 정의
features = [
    "store_nbr", "onpromotion", "oil_price", "year", "month", "day",
    "dayofweek", "is_weekend", "is_payday", "is_holiday", "lag_16", "lag_30", "ewm_sales_7"
]
cat_features = ["store_nbr", "month", "dayofweek"]

# LightGBM 범주형 파라미터 연동을 위한 category 타입 변환
for c in cat_features:
    train_df[c] = train_df[c].astype("category")
    test_df[c] = test_df[c].astype("category")

# Shift 연산으로 발생한 초반 결측치(NaN) 제거
fit_data = train_df.dropna(subset=["lag_16", "lag_30", "ewm_sales_7"])

X_train = fit_data[features]
y_train = fit_data["log_sales"]

# LightGBM 회귀 모델 정의 및 범주형 피처 지정을 통한 학습
model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train, categorical_feature=cat_features)

# [수정] train_df에서 이미 계산된 lag_16, lag_30의 마지막 시점 값만 안전하게 추출
last_lags = train_df.groupby(["store_nbr", "family"], observed=False)[["lag_16", "lag_30"]].last().reset_index()

# test_df에 병합 및 EWMA 추정값 생성
test_df = test_df.merge(last_lags, on=["store_nbr", "family"], how="left")
test_df["ewm_sales_7"] = test_df["lag_16"]

# 예측 수행 및 로그 역변환 (exp(x) - 1), 음수 값 제어 (0 처리)
preds_log = model.predict(test_df[features])
test_df["sales"] = np.expm1(np.maximum(preds_log, 0))

# Kaggle 제출 양식에 맞춘 ID 기준 정렬 및 파일 저장
submission = test_df[["id", "sales"]].sort_values("id")
sub_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(sub_path, index=False)

print(f"[Gayoon] 예측 및 파일 저장 완료: {sub_path}")

Cloning into 'SRL-Code-Review-Study'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 30 (delta 8), reused 6 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 18.00 KiB | 658.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.372885 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1343
[LightGBM] [Info] Number of data points in the train set: 2947428, number of used features: 13
[LightGBM] [Info] Start training from score 2.942922
 예측 및 파일 저장 완료: /content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Gayoon/outputs/submission.csv
